# Stage 3 - Statistical Validation

This notebook validates a small number of already-promoted Stage 2 / Stage 3 findings.
The goal is not broad hypothesis mining. The goal is to strengthen the public claims that already appear in the repository narrative.


## Validation Scope

The notebook checks four focused hypotheses:

1. `Saturday` has higher `totalSteps` than `Sunday`.
2. Higher `awakeAverageStressLevel (D)` is associated with lower `next-night sleepRecoveryScore (D+1)`.
3. Higher `awakeAverageStressLevel (D)` is associated with higher `next-night sleepAverageStressLevel (D+1)`.
4. `Tuesday` tends to have higher awake stress than other weekdays.

Methodological rules:
- day-level weekly comparisons use the strict-quality day slice anchored to `calendarDate`
- day-to-next-night tests use the same `D -> D+1` alignment as Stage 2 directional analysis and Stage 3 modeling
- sleep-onset anchoring is prepared in the stats layer, but is not the main target of this notebook


In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

repo_root = Path.cwd().resolve()
if not (repo_root / 'src').exists():
    repo_root = repo_root.parent
if str(repo_root / 'src') not in sys.path:
    sys.path.insert(0, str(repo_root / 'src'))

from garmin_analytics.stats.prepare import load_stat_validation_frames
from garmin_analytics.stats.inference import (
    kruskal_summary,
    mann_whitney_summary,
    spearman_summary,
)

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 180)


In [2]:
daily_path = repo_root / 'data' / 'processed' / 'daily_sanitized.parquet'
quality_path = repo_root / 'data' / 'processed' / 'daily_quality.parquet'
BOOT_N = 4000
RANDOM_STATE = 42

print('repo_root:', repo_root)
print('daily_path exists:', daily_path.exists(), daily_path)
print('quality_path exists:', quality_path.exists(), quality_path)
print('bootstrap iterations:', BOOT_N)
print('random_state:', RANDOM_STATE)


repo_root: /Users/abatrakov/Documents/FUN/wearable-analytics
daily_path exists: True /Users/abatrakov/Documents/FUN/wearable-analytics/data/processed/daily_sanitized.parquet
quality_path exists: True /Users/abatrakov/Documents/FUN/wearable-analytics/data/processed/daily_quality.parquet
bootstrap iterations: 4000
random_state: 42


In [3]:
frames = load_stat_validation_frames(daily_path, quality_path)

day_strict = frames['day_strict'].copy()
sleep_onset = frames['sleep_onset'].copy()
day_nextsleep = frames['day_nextsleep'].copy()

summary = pd.DataFrame(
    [
        {
            'rows_day_strict': len(day_strict),
            'rows_sleep_onset': len(sleep_onset),
            'rows_day_nextsleep': len(day_nextsleep),
            'rows_with_next_recovery': int(pd.to_numeric(day_nextsleep['nextsleep_sleepRecoveryScore'], errors='coerce').notna().sum()),
            'rows_with_next_sleep_stress': int(pd.to_numeric(day_nextsleep['nextsleep_sleepAverageStressLevel'], errors='coerce').notna().sum()),
        }
    ]
)

display(summary)


,rows_day_strict,rows_sleep_onset,rows_day_nextsleep,rows_with_next_recovery,rows_with_next_sleep_stress
0,525,474,525,464,451


## H1. Saturday vs Sunday steps

Public claim being checked:
- Saturday is the most active day, while Sunday is the clearest low-activity day.

Primary test:
- `Mann-Whitney U` on `totalSteps` with directional alternative `Saturday > Sunday`

Supporting output:
- median difference with bootstrap CI
- Cliff's delta effect size


In [4]:
sat_steps = day_strict.loc[day_strict['weekday_name'] == 'Saturday', 'totalSteps']
sun_steps = day_strict.loc[day_strict['weekday_name'] == 'Sunday', 'totalSteps']
sat_active = day_strict.loc[day_strict['weekday_name'] == 'Saturday', 'active_hours']
sun_active = day_strict.loc[day_strict['weekday_name'] == 'Sunday', 'active_hours']

h1_steps = mann_whitney_summary(
    sat_steps,
    sun_steps,
    alternative='greater',
    n_boot=BOOT_N,
    random_state=RANDOM_STATE,
)
h1_active = mann_whitney_summary(
    sat_active,
    sun_active,
    alternative='greater',
    n_boot=BOOT_N,
    random_state=RANDOM_STATE,
)

h1_df = pd.DataFrame([
    {'metric': 'totalSteps', **h1_steps},
    {'metric': 'active_hours', **h1_active},
])
display(h1_df)


,metric,test,alternative,n_x,n_y,median_x,median_y,median_diff,median_diff_ci_low,median_diff_ci_high,statistic_u,p_value,cliffs_delta
0,totalSteps,mannwhitneyu,greater,75,74,8079.000000,2821.500000,5257.500000,2606.475000,8348.000000,3787.0,0.000061,0.364685
1,active_hours,mannwhitneyu,greater,75,74,1.351111,0.496389,0.854722,0.346667,1.283889,3806.0,0.000046,0.371532


## H2. Awake stress (D) vs next-night recovery (D+1)

Public claim being checked:
- higher daytime stress tends to precede weaker next-night recovery

Primary test:
- `Spearman` correlation with directional alternative `less`

Interpretation target:
- a negative monotonic association is enough to support the public narrative


In [5]:
h2 = spearman_summary(
    day_nextsleep['awakeAverageStressLevel'],
    day_nextsleep['nextsleep_sleepRecoveryScore'],
    alternative='less',
    n_boot=BOOT_N,
    random_state=RANDOM_STATE,
)

h2_df = pd.DataFrame([
    {
        'predictor': 'awakeAverageStressLevel',
        'outcome': 'nextsleep_sleepRecoveryScore',
        **h2,
    }
])
display(h2_df)


,predictor,outcome,test,alternative,n,rho,p_value,rho_ci_low,rho_ci_high
0,awakeAverageStressLevel,nextsleep_sleepRecoveryScore,spearmanr,less,464,-0.284973,2.025476e-10,-0.365639,-0.197339


## H3. Awake stress (D) vs next-night sleep stress (D+1)

Public claim being checked:
- stress propagates beyond the calendar day and is visible in the following night as well

Primary test:
- `Spearman` correlation with directional alternative `greater`


In [6]:
h3 = spearman_summary(
    day_nextsleep['awakeAverageStressLevel'],
    day_nextsleep['nextsleep_sleepAverageStressLevel'],
    alternative='greater',
    n_boot=BOOT_N,
    random_state=RANDOM_STATE,
)

h3_df = pd.DataFrame([
    {
        'predictor': 'awakeAverageStressLevel',
        'outcome': 'nextsleep_sleepAverageStressLevel',
        **h3,
    }
])
display(h3_df)


,predictor,outcome,test,alternative,n,rho,p_value,rho_ci_low,rho_ci_high
0,awakeAverageStressLevel,nextsleep_sleepAverageStressLevel,spearmanr,greater,451,0.266123,4.741344e-09,0.179235,0.350779


## H4. Tuesday higher stress than other weekdays

This is a secondary weekly-rhythm check.
It is interesting, but weaker as a headline claim than H1-H3.

Tests:
- omnibus `Kruskal-Wallis` across weekdays
- planned contrast `Tuesday > non-Tuesday` using `Mann-Whitney U`


In [7]:
weekday_groups = {
    name: group['awakeAverageStressLevel']
    for name, group in day_strict.groupby('weekday_name', observed=False)
}

h4_omnibus = kruskal_summary(weekday_groups)

tuesday_stress = day_strict.loc[day_strict['weekday_name'] == 'Tuesday', 'awakeAverageStressLevel']
non_tuesday_stress = day_strict.loc[day_strict['weekday_name'] != 'Tuesday', 'awakeAverageStressLevel']
h4_contrast = mann_whitney_summary(
    tuesday_stress,
    non_tuesday_stress,
    alternative='greater',
    n_boot=BOOT_N,
    random_state=RANDOM_STATE,
)

display(pd.DataFrame([h4_omnibus]))
display(pd.DataFrame([h4_contrast]))


,test,group_count,statistic_h,p_value,n_Friday,median_Friday,n_Monday,median_Monday,n_Saturday,median_Saturday,n_Sunday,median_Sunday,n_Thursday,median_Thursday,n_Tuesday,median_Tuesday,n_Wednesday,median_Wednesday
0,kruskal,7,5.416841,0.491564,70,55.5,76,54.0,75,56.0,74,51.5,77,55.0,75,58.0,78,52.0


,test,alternative,n_x,n_y,median_x,median_y,median_diff,median_diff_ci_low,median_diff_ci_high,statistic_u,p_value,cliffs_delta
0,mannwhitneyu,greater,75,450,58.0,54.0,4.0,-4.0,8.0,18591.0,0.079134,0.101689


## Summary Table

This table is intended to be the compact promotion layer into `docs/stage3.md` and, if useful, into the case study.


In [8]:
summary_rows = [
    {
        'hypothesis': 'Saturday > Sunday totalSteps',
        'slice': 'day_strict',
        'test': h1_steps['test'],
        'n': f"{h1_steps['n_x']} vs {h1_steps['n_y']}",
        'effect': f"Cliff\'s delta={h1_steps['cliffs_delta']:.3f}",
        'estimate': f"median diff={h1_steps['median_diff']:.1f}",
        'ci': f"[{h1_steps['median_diff_ci_low']:.1f}, {h1_steps['median_diff_ci_high']:.1f}]",
        'p_value': h1_steps['p_value'],
        'interpretation': 'supported' if h1_steps['p_value'] < 0.05 else 'weak / inconclusive',
    },
    {
        'hypothesis': 'Awake stress (D) -> lower next-night recovery',
        'slice': 'day_nextsleep',
        'test': h2['test'],
        'n': int(h2['n']),
        'effect': f"rho={h2['rho']:.3f}",
        'estimate': f"rho={h2['rho']:.3f}",
        'ci': f"[{h2['rho_ci_low']:.3f}, {h2['rho_ci_high']:.3f}]",
        'p_value': h2['p_value'],
        'interpretation': 'supported' if h2['p_value'] < 0.05 else 'weak / inconclusive',
    },
    {
        'hypothesis': 'Awake stress (D) -> higher next-night sleep stress',
        'slice': 'day_nextsleep',
        'test': h3['test'],
        'n': int(h3['n']),
        'effect': f"rho={h3['rho']:.3f}",
        'estimate': f"rho={h3['rho']:.3f}",
        'ci': f"[{h3['rho_ci_low']:.3f}, {h3['rho_ci_high']:.3f}]",
        'p_value': h3['p_value'],
        'interpretation': 'supported' if h3['p_value'] < 0.05 else 'weak / inconclusive',
    },
    {
        'hypothesis': 'Tuesday higher awake stress than other weekdays',
        'slice': 'day_strict',
        'test': 'kruskal + mannwhitneyu',
        'n': f"Tue {h4_contrast['n_x']} vs other {h4_contrast['n_y']}",
        'effect': f"Cliff\'s delta={h4_contrast['cliffs_delta']:.3f}",
        'estimate': f"median diff={h4_contrast['median_diff']:.1f}",
        'ci': f"[{h4_contrast['median_diff_ci_low']:.1f}, {h4_contrast['median_diff_ci_high']:.1f}]",
        'p_value': h4_contrast['p_value'],
        'interpretation': 'supported' if (h4_omnibus['p_value'] < 0.05 and h4_contrast['p_value'] < 0.05) else 'weak / inconclusive',
    },
]

summary_table = pd.DataFrame(summary_rows)
display(summary_table)


,hypothesis,slice,test,n,effect,estimate,ci,p_value,interpretation
0,Saturday > Sunday totalSteps,day_strict,mannwhitneyu,75 vs 74,Cliff's delta=0.365,median diff=5257.5,"[2606.5, 8348.0]",6.144284e-05,supported
1,Awake stress (D) -> lower next-night recovery,day_nextsleep,spearmanr,464,rho=-0.285,rho=-0.285,"[-0.366, -0.197]",2.025476e-10,supported
2,Awake stress (D) -> higher next-night sleep st...,day_nextsleep,spearmanr,451,rho=0.266,rho=0.266,"[0.179, 0.351]",4.741344e-09,supported
3,Tuesday higher awake stress than other weekdays,day_strict,kruskal + mannwhitneyu,Tue 75 vs other 450,Cliff's delta=0.102,median diff=4.0,"[-4.0, 8.0]",7.913427e-02,weak / inconclusive
